In [2]:
%pip install pandas
%pip install wordninja
%pip install deep-translator
%pip install langdetect
%pip install nltk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import wordninja
import os
import string
import langdetect
import time
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from langdetect import detect
from deep_translator import GoogleTranslator


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package punkt to C:\Users\Lish Ai
[nltk_data]     Labs\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [7]:
#load dataset
file_path = 'C:/Users/Lish Ai Labs/Desktop/Simba/warfare_data.csv'
df = pd.read_csv(file_path)
print(df.head())


                                   title  \
0                                    War   
1                      Guerrilla warfare   
2                         Trench warfare   
3  The Ministry of Ungentlemanly Warfare   
4                        Nuclear warfare   

                                             summary  \
0                                                NaN   
1  Guerrilla warfare is a form of unconventional ...   
2  \nTrench warfare is a  type of land warfare us...   
3  \nThe Ministry of Ungentlemanly Warfare is a 2...   
4  Nuclear warfare, also known as atomic warfare,...   

                                             content  \
0                                                NaN   
1  Prehistoric\nAncient\nPost-classical\nCastles\...   
2  \nPrehistoric\nAncient\nPost-classical\nCastle...   
3  \nPaul Tamasy\nEric Johnson\nArash Amel\nGuy R...   
4  Prehistoric\nAncient\nPost-classical\nCastles\...   

                                               links  \
0    

In [26]:
print(df.dtypes)

title      object
summary    object
content    object
links      object
url        object
dtype: object


In [27]:
# checking for missing values
missing_values = df.isnull()
for column in missing_values.columns.values.tolist():
    print (missing_values[column].value_counts())
    print("")


title
False    9998
Name: count, dtype: int64

summary
False    8629
True     1369
Name: count, dtype: int64

content
False    8636
True     1362
Name: count, dtype: int64

links
False    9998
Name: count, dtype: int64

url
False    9998
Name: count, dtype: int64



In [28]:
#display the duplicate rows
duplicate_rows_df = df[df.duplicated()]
print("number of duplicate rows: ", duplicate_rows_df.shape)

number of duplicate rows:  (1478, 5)


In [29]:
#dropping duplicates
df = df.drop_duplicates()
print("number of duplicate rows: ", df.duplicated())

number of duplicate rows:  0       False
1       False
2       False
3       False
4       False
        ...  
9993    False
9994    False
9995    False
9996    False
9997    False
Length: 8520, dtype: bool


In [30]:
# Drop rows where both columns 'content' and 'summary' are empty
df1 = df.copy()
df_cleaned = df1.dropna(subset=['content', 'summary'], how='all')

In [31]:
#checking for missing values after removing duplicates
missing_valuesdf1 = df_cleaned.isnull()
for column in missing_valuesdf1.columns.values.tolist():
    print (missing_valuesdf1[column].value_counts())
    print("")

title
False    7352
Name: count, dtype: int64

summary
False    7345
True        7
Name: count, dtype: int64

content
False    7352
Name: count, dtype: int64

links
False    7352
Name: count, dtype: int64

url
False    7352
Name: count, dtype: int64



In [32]:
# having only 7 rows with missing we can replace them with Not Available

df1 = df.copy()

df_cleaned = df1.dropna(subset=['summary'], how='all').copy()

# Function to extract first 25 words
def get_summary(text):
    words = text.split()
    return ' '.join(words[:25]) if len(words) > 25 else text

df_cleaned.loc[df_cleaned['summary'].isna(), 'summary'] = df_cleaned.loc[df_cleaned['summary'].isna(), 'content'].apply(get_summary)


In [33]:
# checking if there are any missing values left
missing_valuesdf1 = df_cleaned.isnull()
for column in missing_valuesdf1.columns.values.tolist():
    print (missing_valuesdf1[column].value_counts())
    print("")

title
False    7345
Name: count, dtype: int64

summary
False    7345
Name: count, dtype: int64

content
False    7345
Name: count, dtype: int64

links
False    7345
Name: count, dtype: int64

url
False    7345
Name: count, dtype: int64



In [34]:
#removing \n from the content table and summary
df_cleaned["content"] = df_cleaned["content"].str.replace("\n", " ", regex=True)
df_cleaned["summary"] = df_cleaned["summary"].str.replace("\n", " ", regex=True)

In [35]:
df_cleaned.head()


,title,summary,content,links,url
1,Guerrilla warfare,Guerrilla warfare is a form of unconventional ...,Prehistoric Ancient Post-classical Castles Cas...,"['/wiki/Swarming_(military)', '/wiki/Stay-behi...",https://en.wikipedia.org/wiki/Guerrilla_warfare
2,Trench warfare,Trench warfare is a type of land warfare usi...,Prehistoric Ancient Post-classical Castles Ca...,"['/wiki/Dolomites', '/wiki/101st_Airborne_Divi...",https://en.wikipedia.org/wiki/Trench_warfare
3,The Ministry of Ungentlemanly Warfare,The Ministry of Ungentlemanly Warfare is a 20...,Paul Tamasy Eric Johnson Arash Amel Guy Ritch...,"['/wiki/Operation_Fortune:_Ruse_de_Guerre', '/...",https://en.wikipedia.org/wiki/The_Ministry_of_...
4,Nuclear warfare,"Nuclear warfare, also known as atomic warfare,...",Prehistoric Ancient Post-classical Castles Cas...,"['/wiki/Conflict_escalation', '/wiki/Deforesta...",https://en.wikipedia.org/wiki/Nuclear_warfare
5,Chemical warfare,Chemical warfare (CW) involves using the toxi...,Cyanogen chloride (CK) Hydrogen cyanide (AC) ...,"['/wiki/Portal:Environment', '/wiki/Future_of_...",https://en.wikipedia.org/wiki/Chemical_warfare


In [11]:
# normalizing text
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_dataframe(df):
    """Normalizes all string columns in a Pandas DataFrame."""
    for col in df.columns:
        if df[col].dtype == 'object':  
            try:
                df[col] = df[col].astype(str).apply(normalize_text) 
            except Exception as e:
                print(f"Error normalizing column {col}: {e}")
    return df

try:
    df = pd.read_csv(file_path)  

    # Normalize the entire DataFrame
    df = normalize_dataframe(df)

    # Print the first few rows to check the results
    print(df.head())

except FileNotFoundError:
    print("Error: CSV file not found.  Make sure the file path is correct.")
except Exception as e:
    print(f"An error occurred: {e}")

                                   title  \
0                                    war   
1                      guerrilla warfare   
2                         trench warfare   
3  the ministry of ungentlemanly warfare   
4                        nuclear warfare   

                                             summary  \
0                                                nan   
1  guerrilla warfare is a form of unconventional ...   
2  trench warfare is a type of land warfare using...   
3  the ministry of ungentlemanly warfare is a 202...   
4  nuclear warfare also known as atomic warfare i...   

                                             content  \
0                                                nan   
1  prehistoric ancient postclassical castles cast...   
2  prehistoric ancient postclassical castles cast...   
3  paul tamasy eric johnson arash amel guy ritchi...   
4  prehistoric ancient postclassical castles cast...   

                                               links  \
0    

In [15]:
# 
df_cleaned = pd.read_csv(file_path)
print("DataFrame loaded successfully.") 
if 'summary' not in df_cleaned.columns or 'content' not in df_cleaned.columns:
            raise ValueError("The DataFrame must have 'summary' and 'content' columns.")
print("Columns 'summary' and 'content' verified.") 

df_cleaned['summary'] = df_cleaned['summary'].astype(str).apply(lambda x: " ".join(wordninja.split(x)))
print("Word splitting applied to 'summary' column.") 
df_cleaned['content'] = df_cleaned['content'].astype(str).apply(lambda x: " ".join(wordninja.split(x)))
print("Word splitting applied to 'content' column.") 

print("DataFrame processing completed successfully.") 


DataFrame loaded successfully.
Columns 'summary' and 'content' verified.
Word splitting applied to 'summary' column.
Word splitting applied to 'content' column.
DataFrame processing completed successfully.


In [16]:
df_cleaned['links'] = df['links']

df_cleaned.head()

,title,summary,content,links,url
0,War,nan,nan,wikiwikipediaprotectionpolicysemi,https://en.wikipedia.org/wiki/War
1,Guerrilla warfare,Guerrilla warfare is a form of unconventional ...,Prehistoric Ancient Post classical Castles Cas...,wikiswarmingmilitary wikistaybehind wikicavalr...,https://en.wikipedia.org/wiki/Guerrilla_warfare
2,Trench warfare,Trench warfare is a type of land warfare using...,Prehistoric Ancient Post classical Castles Cas...,wikidolomites wiki101stairbornedivision wikitr...,https://en.wikipedia.org/wiki/Trench_warfare
3,The Ministry of Ungentlemanly Warfare,The Ministry of Ungentlemanly Warfare is a 202...,Paul Tamas y Eric Johnson A rash Amel Guy Ritc...,wikioperationfortunerusedeguerre wikichristoph...,https://en.wikipedia.org/wiki/The_Ministry_of_...
4,Nuclear warfare,Nuclear warfare also known as atomic warfare i...,Prehistoric Ancient Post classical Castles Cas...,wikiconflictescalation wikideforestation wikis...,https://en.wikipedia.org/wiki/Nuclear_warfare


In [17]:
# removing links from the summary, content and title columns


def remove_url(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

#This function removes punctuations
def remove_punct(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df_cleaned['content'] = df_cleaned['content'].apply(lambda x: remove_url(x))
df_cleaned['summary'] = df_cleaned['summary'].apply(lambda x: remove_url(x))
df_cleaned['title'] = df_cleaned['title'].apply(lambda x: remove_url(x))

df_cleaned = df_cleaned.drop(columns=['links'])

In [6]:
# Checking for stopwords
try:
    stopwords.words('english')
    print("Stopwords already downloaded.")
except LookupError:
    print("Downloading stopwords...")
    nltk.download('stopwords')

# Checking for punkt tokenizer
try:
    word_tokenize("test")
    print("Punkt tokenizer already downloaded.")
except LookupError:
    print("Downloading punkt tokenizer...")
    nltk.download('punkt')

Stopwords already downloaded.


[nltk_data] Downloading package punkt to C:\Users\Lish Ai
[nltk_data]     Labs\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
